# DSE 200 — Day 4 Lab, part 3
## Interpretability: asking a model why

**Sections 10.1–10.5 match the slides · agent on: explain every cell.**

| Cell | What to do |
| --- | --- |
| `▶ DEMO` | Already written. Run it while we talk. |
| `🔍 READ` | **Predict before running.** Commit to an answer. |
| `✍️ YOU TRY` | Has a `# TODO`. Yours. |
| `✅ CHECKPOINT` | Run it. It tells you whether you are right. |

**Open the table of contents** (☰, left edge). *Runtime → Run all* if you get lost.

In [ ]:
!uv pip install --system -q shap

In [ ]:
# ▶ SETUP — run first. Rebuilds the Gradient Boosting notebook's data and model (about a minute).
import os, time, textwrap, urllib.request
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
import shap
def find(name):
    """Return a local path for `name`, downloading it from the course repo if needed."""
    for p in [name, "/content/" + name, "data/" + name, "../" + name]:
        if os.path.exists(p):
            return p
    try:
        urllib.request.urlretrieve("https://raw.githubusercontent.com/DSE200-2026/DSE200/main/" + name, name)
        print("downloaded", name, "from the course repo")
        return name
    except Exception as e:
        raise FileNotFoundError(f"{name} not found -- drag it into the Files pane (folder icon, left)") from e

df = pd.read_csv(find("minute_weather.csv.gz"), parse_dates=["hpwren_timestamp"])
df = df.drop_duplicates(subset=df.columns.drop("rowID"))            # the stuck logger (8.3)
df = df.sort_values("hpwren_timestamp").reset_index(drop=True)
df["dry_now"] = (df["relative_humidity"] < 25).astype(int)
df["dry_3h"]  = (df["relative_humidity"].shift(-180) < 25).astype(int)   # the label (8.4)
df = df.iloc[:-180].copy()

# features that look only backward (8.7)
df["hour"]  = df["hpwren_timestamp"].dt.hour
df["month"] = df["hpwren_timestamp"].dt.month
df["rh_change_1h"] = df["relative_humidity"] - df["relative_humidity"].shift(60)
df["rh_change_3h"] = df["relative_humidity"] - df["relative_humidity"].shift(180)
df["temp_change_1h"] = df["air_temp"] - df["air_temp"].shift(60)
df["pressure_change_3h"] = df["air_pressure"] - df["air_pressure"].shift(180)
sensors = ["air_pressure", "air_temp", "avg_wind_direction", "avg_wind_speed",
           "max_wind_direction", "max_wind_speed", "min_wind_direction",
           "min_wind_speed", "rain_accumulation", "rain_duration", "relative_humidity"]
features = sensors + ["hour", "month", "rh_change_1h", "rh_change_3h",
                      "temp_change_1h", "pressure_change_3h"]

# year 1 trains, year 2 chooses settings, year 3 is the final test (8.5)
year1 = df[df["hpwren_timestamp"] < "2012-09-10"]
year2 = df[(df["hpwren_timestamp"] >= "2012-09-10") & (df["hpwren_timestamp"] < "2013-09-10")]
year3 = df[df["hpwren_timestamp"] >= "2013-09-10"]
model = XGBClassifier(n_estimators=3000, learning_rate=0.05, max_depth=3,
                      early_stopping_rounds=20, random_state=42)
model.fit(year1[features], year1["dry_3h"],
          eval_set=[(year2[features], year2["dry_3h"])], verbose=False)

_READ = {
"10.2": ("7 of 18. Mid-table, ahead of real sensors.",
         "Split-count importance counts how often a column was used to split, on the TRAINING rows.\n"
         "A column of random decimals offers a fresh threshold at every value, so a deep model\n"
         "keeps finding splits on it that happen to help on rows it has already seen.\n"
         "Permutation importance on held-out rows gives the same column about zero."),
"10.5": ("rh_next_hour_avg, by a mile: about 0.34, when the next feature is under 0.01.",
         "It is the mean humidity two to three hours AHEAD: it encodes the answer.\n"
         "A feature that is implausibly important is usually a leak. This is how you catch one in ten seconds."),
}

def check_read(tag, prediction):
    if not str(prediction).strip() or str(prediction).strip() in ("???", "?"):
        print("Write a prediction first. Committing to a wrong answer is how this works.")
        return
    ans, why = _READ[tag]
    print("your prediction:"); print(textwrap.indent(str(prediction), "    "))
    print("\nactual:");        print(textwrap.indent(ans, "    "))
    print("\n" + why)

print(f"{len(df):,} minutes. Model trained; check_read() ready.")

# The Question You Will Be Asked

*Slides: Interpretability · section 10.1*

## 10.1 "Why Did It Say That?"

By a manager, a regulator, or a customer. You need an answer that does not begin with "the algorithm".

In [ ]:
# ▶ DEMO -- one minute the model is fairly sure about
test = year3.sample(20000, random_state=0)       # 20,000 year-3 minutes: same answers, 25x faster
prob = model.predict_proba(test[features])[:, 1]
i = np.argmin(np.abs(prob - 0.83))
print(test["hpwren_timestamp"].iloc[i], "  P(dry in 3 h) =", round(prob[i], 2))
test[features].iloc[i]

# The Built-In Answer

*Slides: Interpretability · section 10.2*

## 10.2 `feature_importances_`

Free, instant, and the first thing everyone reaches for. It counts how a feature was used to split, **on the training data**.

In [ ]:
# ▶ DEMO
pd.Series(model.feature_importances_, index=features).sort_values().plot.barh(figsize=(7, 5))
plt.title("built-in importance"); plt.show()

In [ ]:
# 🔍 READ 10.2 -- the two-line experiment. Add a column of pure random numbers.
#
#     A deep XGBoost (max_depth=10), importance = how often each column is used to split.
#     Out of 18 columns, where does random_noise rank? Commit to a number.

prediction = "???"
check_read("10.2", prediction)

In [ ]:
# ▶ DEMO -- see it
small = year1.sample(30000, random_state=1).copy()
small["random_noise"] = np.random.default_rng(0).normal(size=len(small))
cols = features + ["random_noise"]

deep = XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=10, importance_type="weight", random_state=0)
deep.fit(small[cols], small["dry_3h"])
imp = pd.Series(deep.feature_importances_, index=cols).sort_values(ascending=False)
print("random_noise ranks", list(imp.index).index("random_noise") + 1, "of", len(cols))
imp.round(3)

# Permutation Importance

*Slides: Interpretability · section 10.3*

## 10.3 The Honest Version

Take a trained model and a **test** set. Shuffle one column. Score it again. **However much the score dropped is what that feature was worth.** It never looks inside the model; it only needs `predict`.

In [ ]:
# ▶ DEMO -- shuffle one column by hand
shuffled = test[features].copy()
shuffled["relative_humidity"] = np.random.default_rng(0).permutation(shuffled["relative_humidity"].values)
print("as is              ", round(model.score(test[features], test["dry_3h"]), 3))
print("humidity shuffled  ", round(model.score(shuffled, test["dry_3h"]), 3))

In [ ]:
# ▶ DEMO -- every column, 10 shuffles each, on held-out rows
from sklearn.inspection import permutation_importance

r = permutation_importance(model, test[features], test["dry_3h"], n_repeats=10, random_state=0)
perm = pd.Series(r.importances_mean, index=features).sort_values()
perm.plot.barh(xerr=pd.Series(r.importances_std, index=features)[perm.index], figsize=(7, 5))
plt.title("drop in accuracy when shuffled"); plt.show()

In [ ]:
# ▶ DEMO -- the same code on a completely different model
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

sample = year1.sample(50000, random_state=2)
logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
logreg.fit(sample[features].fillna(0), sample["dry_3h"])
r = permutation_importance(logreg, test[features].fillna(0), test["dry_3h"], n_repeats=5, random_state=0)
pd.Series(r.importances_mean, index=features).sort_values(ascending=False).head(5).round(3)

**One trap: correlated features.** If two columns carry the same information, shuffling one changes little: the model leans on the other. Below, we add a near-copy of humidity and retrain.

In [ ]:
# ▶ DEMO -- two columns, one piece of information
rng = np.random.default_rng(0)
y1c, y2c, tc = year1.copy(), year2.copy(), test.copy()
for d in (y1c, y2c, tc):
    d["humidity_copy"] = d["relative_humidity"] + rng.normal(0, 0.3, len(d))
cols = features + ["humidity_copy"]
m2 = XGBClassifier(n_estimators=3000, learning_rate=0.05, max_depth=3, early_stopping_rounds=20, random_state=42)
m2.fit(y1c[cols], y1c["dry_3h"], eval_set=[(y2c[cols], y2c["dry_3h"])], verbose=False)
r = permutation_importance(m2, tc[cols], tc["dry_3h"], n_repeats=10, random_state=0)
pd.Series(r.importances_mean, index=cols)[["relative_humidity", "humidity_copy"]].round(3)

The copy scores about **zero**, and it holds exactly the same information as the most important column in the file. **Check your correlation matrix before you believe a zero.**

# SHAP

*Slides: Interpretability · section 10.4*

## 10.4 Per Row, Not Per Model

Permutation importance: which features matter, overall? **SHAP: for this one row, what pushed the answer up and what pushed it down?** Nobody was ever denied a loan by an average.

In [ ]:
# ▶ DEMO -- the summary plot: one dot per minute, one row per feature
explainer = shap.TreeExplainer(model)
rows = test.sample(2000, random_state=3)[features]
values = explainer.shap_values(rows)
shap.summary_plot(values, rows, max_display=10)

**How to read it:** most important feature at the top. **Right** pushes the prediction up, **left** pushes it down. Colour is the feature's value: red high, blue low. So a blue band on the right of `relative_humidity` means "low humidity now pushes toward dry later".

In [ ]:
# ▶ DEMO -- one minute's answer, explained
shap.plots.waterfall(explainer(test[features].iloc[[i]])[0], max_display=8)

In [ ]:
# ▶ DEMO -- SHAP on a model that is not a tree: the logistic regression from 10.3
X = test[features].fillna(0)
f = lambda d: logreg.predict_proba(pd.DataFrame(d, columns=features))[:, 1]   # any model with predict_proba
background = shap.maskers.Independent(X.sample(100, random_state=0))
explainer_any = shap.Explainer(f, background, feature_names=features)
shap.plots.waterfall(explainer_any(X.iloc[[i]])[0], max_display=8)

Same minute, a different model. Compare it with the XGBoost waterfall above: both lean on humidity first; below that they disagree. **Two models can agree on the answer for different reasons.**

In [ ]:
# ▶ DEMO -- dependence plot: each minute's humidity, against what humidity did to its prediction
sv_tree = explainer(rows)
shap.plots.scatter(sv_tree[:, "relative_humidity"], color=sv_tree[:, "hour"])

In [ ]:
# ▶ DEMO -- partial dependence, from sklearn: no extra install, works on any model
from sklearn.inspection import PartialDependenceDisplay

PartialDependenceDisplay.from_estimator(model, test[features].sample(2000, random_state=4).astype(float),
                                        ["relative_humidity", "hour"], kind="both", subsample=60, random_state=0)
plt.show()

**Importance is not causation.** Every method here tells you what the **model** leaned on. `month` is predictive; changing the month will not dry the air. A feature can be important purely because it correlates with something you did not measure.

# Use It The Other Way Round

*Slides: Interpretability · section 10.5*

## 10.5 A Leakage Alarm

A feature that is implausibly important is usually a feature that encodes the answer.

In [ ]:
# 🔍 READ 10.5 -- below, someone adds rh_next_hour_avg: "average humidity over the next hour block".
#
#     Run permutation importance. Which column comes out on top, and by how much? Commit first.

prediction = "???"
check_read("10.5", prediction)

In [ ]:
# ✍️ YOU TRY 10.5
# The leaky column: mean humidity from 2 to 3 hours AHEAD.
df["rh_next_hour_avg"] = df["relative_humidity"].shift(-120).rolling(60).mean()
y1l = df.loc[year1.index]; y2l = df.loc[year2.index]; tl = df.loc[test.index]
cols = features + ["rh_next_hour_avg"]

leaky = None    # TODO: fit the same XGBoost (early stopping on y2l) on cols
leak_imp = None # TODO: permutation importance of leaky on tl[cols], as a pd.Series indexed by cols

In [ ]:
# ✅ CHECKPOINT 10.5
assert leaky is not None and leak_imp is not None, "fit the model and compute permutation importance"
top = leak_imp.sort_values(ascending=False)
print(top.head(4).round(3))
assert top.index[0] == "rh_next_hour_avg", "the leak should come out on top"
print("checkpoint passed: one bar towers over the rest. That is the alarm.")

### ✍️ In one sentence
**You are shown a model whose top feature is 30 times more important than the next one. What do you check first?**

*Your answer:*